# Robust trench-image rotation

This notebook translates `ITI_Profiling_Align_RMQA_FINAL.m` into a deterministic, class-friendly Python API. The core function accepts either two left-to-right trench points or an explicit angle, applies MATLAB-compatible fixed-size rotation, and returns calibrated coordinates. An optional interactive wrapper reproduces the four two-click selections in the MATLAB workflow.

Coordinate convention: points are `(x, y) = (column, row)` in pixels, with `x` increasing rightward and `y` increasing downward. Therefore, a trench that slopes downward to the right has a positive measured angle and is corrected with a positive (counterclockwise) image rotation.

In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional, Sequence, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage


ImageInput = Union[str, Path, np.ndarray]


@dataclass(frozen=True)
class TrenchRotationResult:
    """Output of :func:`rotate_trench_image`."""

    image: np.ndarray
    angle_deg: float
    x_um: np.ndarray
    y_um: np.ndarray
    center_px: Tuple[float, float]
    rotation_points_px: Optional[np.ndarray]

    def save(self, destination: Union[str, Path]) -> None:
        """Save the rotated image without changing its array data."""
        destination = Path(destination).expanduser()
        destination.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(self.image).save(destination)


In [ ]:
def _load_image(image: ImageInput) -> np.ndarray:
    """Load a path or defensively copy an existing numeric image array."""
    if isinstance(image, (str, Path)):
        path = Path(image).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"Image file does not exist: {path}")
        with Image.open(path) as pil_image:
            # Convert palette/bilevel images to explicit channels; preserve RGB(A),
            # grayscale, and higher-bit-depth grayscale arrays as they are.
            if pil_image.mode in {"1", "P"}:
                pil_image = pil_image.convert("RGB")
            array = np.array(pil_image)
    elif isinstance(image, np.ndarray):
        array = np.array(image, copy=True)
    else:
        raise TypeError("image must be a filesystem path or a NumPy array")

    if array.ndim not in (2, 3):
        raise ValueError(f"Expected a 2-D or 3-D image; received shape {array.shape}")
    if array.ndim == 3 and array.shape[2] not in (1, 3, 4):
        raise ValueError(
            "A 3-D image must have 1 (grayscale), 3 (RGB), or 4 (RGBA) channels"
        )
    if array.shape[0] < 1 or array.shape[1] < 1:
        raise ValueError("Image height and width must both be nonzero")
    if not (np.issubdtype(array.dtype, np.integer) or np.issubdtype(array.dtype, np.floating)):
        raise TypeError(f"Image dtype must be real numeric, not {array.dtype}")
    if np.issubdtype(array.dtype, np.floating) and not np.isfinite(array).all():
        raise ValueError("Floating-point image contains NaN or infinity")
    return array


def _add_brightness_safely(image: np.ndarray, offset: float) -> np.ndarray:
    """Add brightness without uint8 wraparound; leave an alpha channel unchanged."""
    if not np.isscalar(offset) or not np.isfinite(offset):
        raise ValueError("brightness_offset must be a finite scalar")

    output = np.array(image, copy=True)
    if float(offset) == 0.0:
        return output

    target = output[..., :3] if output.ndim == 3 and output.shape[2] == 4 else output
    bright = target.astype(np.float64) + float(offset)
    if not np.isfinite(bright).all():
        raise OverflowError("brightness_offset overflowed the image values")
    if np.issubdtype(output.dtype, np.integer):
        limits = np.iinfo(output.dtype)
        bright = np.clip(bright, limits.min, limits.max)
        target[...] = np.rint(bright).astype(output.dtype)
    else:
        target[...] = bright.astype(output.dtype)
    return output


def _validate_points(
    points: Sequence[Sequence[float]],
    *,
    name: str,
    width: int,
    height: int,
) -> np.ndarray:
    array = np.asarray(points, dtype=float)
    if array.shape != (2, 2):
        raise ValueError(f"{name} must contain exactly two (x, y) points; got {array.shape}")
    if not np.isfinite(array).all():
        raise ValueError(f"{name} contains NaN or infinity")
    x, y = array[:, 0], array[:, 1]
    if np.any(x < 0) or np.any(x > width - 1) or np.any(y < 0) or np.any(y > height - 1):
        raise ValueError(
            f"{name} must lie inside image bounds x=[0, {width - 1}], y=[0, {height - 1}]"
        )
    return array


def _coordinate_axes(
    shape: Tuple[int, ...],
    center_px: Tuple[float, float],
    pixel_pitch_um: Sequence[float],
    magnification: float,
) -> Tuple[np.ndarray, np.ndarray]:
    pitch = np.asarray(pixel_pitch_um, dtype=float)
    if pitch.shape != (2,) or not np.isfinite(pitch).all() or np.any(pitch <= 0):
        raise ValueError("pixel_pitch_um must be two positive finite values: (x_pitch, y_pitch)")
    if not np.isscalar(magnification) or not np.isfinite(magnification) or magnification <= 0:
        raise ValueError("magnification must be a positive finite scalar")

    height, width = shape[:2]
    center_x, center_y = center_px
    x_um = (np.arange(width, dtype=float) - center_x) * pitch[0] / float(magnification)
    y_um = (np.arange(height, dtype=float) - center_y) * pitch[1] / float(magnification)
    return x_um, y_um


def _rotate_preserving_dtype(
    image: np.ndarray, angle_deg: float, interpolation_order: int, fill_value: float
) -> np.ndarray:
    # Work in float64 so interpolation behavior is independent of the input integer dtype.
    rotated = ndimage.rotate(
        image.astype(np.float64),
        angle=float(angle_deg),
        axes=(1, 0),
        reshape=False,  # MATLAB imrotate(..., 'crop')
        order=interpolation_order,
        mode="constant",
        cval=float(fill_value),
        prefilter=interpolation_order > 1,
    )
    if np.issubdtype(image.dtype, np.integer):
        limits = np.iinfo(image.dtype)
        return np.rint(np.clip(rotated, limits.min, limits.max)).astype(image.dtype)
    return rotated.astype(image.dtype, copy=False)


def rotate_trench_image(
    image: ImageInput,
    *,
    rotation_points_px: Optional[Sequence[Sequence[float]]] = None,
    angle_deg: Optional[float] = None,
    center_points_px: Optional[Sequence[Sequence[float]]] = None,
    pixel_pitch_um: Sequence[float] = (5.3, 5.3),
    magnification: float = 50.0,
    brightness_offset: float = 50.0,
    interpolation_order: int = 0,
    fill_value: float = 0.0,
) -> TrenchRotationResult:
    """Rotate and optionally recenter a trench image.

    Supply exactly one of ``rotation_points_px`` or ``angle_deg``. Rotation points
    are ordered LEFT then RIGHT in image coordinates. Their angle is computed with
    ``atan2(dy, dx)`` and applied counterclockwise, matching the MATLAB routine.

    ``center_points_px``, when supplied, are two trench-edge points measured on the
    *rotated* image. Their midpoint becomes (x, y) = (0, 0). Otherwise the image
    center is the coordinate origin. The output image keeps the input height, width,
    channel count, and dtype. Order 0 is MATLAB's default nearest-neighbor method;
    use order 1 for bilinear interpolation.
    """
    array = _load_image(image)
    height, width = array.shape[:2]

    if (rotation_points_px is None) == (angle_deg is None):
        raise ValueError("Supply exactly one of rotation_points_px or angle_deg")

    validated_rotation_points = None
    if rotation_points_px is not None:
        validated_rotation_points = _validate_points(
            rotation_points_px, name="rotation_points_px", width=width, height=height
        )
        delta_x, delta_y = validated_rotation_points[1] - validated_rotation_points[0]
        if delta_x <= 0:
            raise ValueError(
                "rotation_points_px must be ordered LEFT then RIGHT (right x must be larger)"
            )
        angle = float(np.degrees(np.arctan2(delta_y, delta_x)))
    else:
        if not np.isscalar(angle_deg) or not np.isfinite(angle_deg):
            raise ValueError("angle_deg must be a finite scalar")
        angle = float(angle_deg)

    if isinstance(interpolation_order, (bool, np.bool_)) or not isinstance(
        interpolation_order, (int, np.integer)
    ):
        raise TypeError("interpolation_order must be an integer from 0 through 5")
    interpolation_order = int(interpolation_order)
    if not 0 <= interpolation_order <= 5:
        raise ValueError("interpolation_order must be from 0 through 5")
    if not np.isscalar(fill_value) or not np.isfinite(fill_value):
        raise ValueError("fill_value must be a finite scalar")

    brightened = _add_brightness_safely(array, brightness_offset)
    rotated = _rotate_preserving_dtype(brightened, angle, interpolation_order, fill_value)

    if center_points_px is None:
        center_px = ((width - 1) / 2.0, (height - 1) / 2.0)
    else:
        center_points = _validate_points(
            center_points_px, name="center_points_px", width=width, height=height
        )
        center_px = tuple(np.mean(center_points, axis=0))

    x_um, y_um = _coordinate_axes(rotated.shape, center_px, pixel_pitch_um, magnification)
    return TrenchRotationResult(
        image=rotated,
        angle_deg=angle,
        x_um=x_um,
        y_um=y_um,
        center_px=(float(center_px[0]), float(center_px[1])),
        rotation_points_px=(
            None if validated_rotation_points is None else validated_rotation_points.copy()
        ),
    )


## Optional MATLAB-style click workflow

The core function above has no GUI dependency and is the part to embed in a class. The wrapper below reproduces the MATLAB sequence: select an ROI, select LEFT/RIGHT orientation points, rotate, select another ROI, then select LEFT/RIGHT centering points. In JupyterLab, enable an interactive backend such as `%matplotlib widget` before calling it.

In [ ]:
def _pick_two_points(
    image: np.ndarray, title: str, roi_points_px: Optional[np.ndarray] = None
) -> np.ndarray:
    fig, ax = plt.subplots(figsize=(12, 8))
    display = image[..., 0] if image.ndim == 3 and image.shape[2] == 1 else image
    ax.imshow(display, origin="upper")
    ax.set_title(title)
    ax.set_xlabel("x / column [px]")
    ax.set_ylabel("y / row [px]")
    if roi_points_px is not None:
        (x0, y0), (x1, y1) = roi_points_px
        ax.set_xlim(sorted((x0, x1)))
        ax.set_ylim(sorted((y0, y1), reverse=True))
    points = np.asarray(plt.ginput(2, timeout=-1, show_clicks=True), dtype=float)
    plt.close(fig)
    if points.shape != (2, 2):
        raise RuntimeError("Point selection was cancelled before two points were chosen")
    return points


def interactive_align_trench_image(
    image: ImageInput,
    *,
    recenter: bool = True,
    pixel_pitch_um: Sequence[float] = (5.3, 5.3),
    magnification: float = 50.0,
    brightness_offset: float = 50.0,
    interpolation_order: int = 0,
    fill_value: float = 0.0,
) -> TrenchRotationResult:
    """Interactive convenience wrapper equivalent to the MATLAB click workflow."""
    source = _load_image(image)
    height, width = source.shape[:2]
    display_image = _add_brightness_safely(source, brightness_offset)

    roi = _pick_two_points(display_image, "Select trench ROI: NW, then SE")
    _validate_points(roi, name="ROI points", width=width, height=height)
    rotation_points = _pick_two_points(
        display_image, "Select trench orientation: LEFT, then RIGHT", roi
    )
    result = rotate_trench_image(
        source,
        rotation_points_px=rotation_points,
        pixel_pitch_um=pixel_pitch_um,
        magnification=magnification,
        brightness_offset=brightness_offset,
        interpolation_order=interpolation_order,
        fill_value=fill_value,
    )

    if not recenter:
        return result

    center_roi = _pick_two_points(result.image, "Select post-rotation trench ROI: NW, then SE")
    _validate_points(center_roi, name="center ROI points", width=width, height=height)
    center_points = _pick_two_points(
        result.image, "Select post-rotation trench edges: LEFT, then RIGHT", center_roi
    )
    center_points = _validate_points(
        center_points, name="center points", width=width, height=height
    )
    center_px = tuple(np.mean(center_points, axis=0))
    x_um, y_um = _coordinate_axes(result.image.shape, center_px, pixel_pitch_um, magnification)
    return replace(
        result,
        x_um=x_um,
        y_um=y_um,
        center_px=(float(center_px[0]), float(center_px[1])),
    )


## Supplied BMP: reproducible non-interactive call

The two example points below follow the clearly visible upper trench edge in `white_light.bmp`. They produce approximately +0.234° of correction. For measurement work, use your selected points (or the interactive wrapper) instead of treating these approximate inspection points as calibration data.

In [ ]:
WHITE_LIGHT_PATH = Path(
    "/Users/yiyangzhi/Library/CloudStorage/GoogleDrive-yiyang_zhi3@berkeley.edu/"
    "My Drive/Ming Wu Integrated Photonics Group/Experiments/Measurements/NANOLAB/"
    "Gen2_Ca_90/I/before_Optelligent/indiv/white_light.bmp"
)

if WHITE_LIGHT_PATH.is_file():
    sample_result = rotate_trench_image(
        WHITE_LIGHT_PATH,
        rotation_points_px=((224.0, 89.256), (970.0, 92.309)),
        # Omit center_points_px to retain the geometric image center as the origin.
        brightness_offset=50.0,  # faithful to the MATLAB source
        interpolation_order=0,  # MATLAB imrotate default: nearest neighbor
    )
    print(f"Applied rotation: {sample_result.angle_deg:.6f} degrees")
    print(f"Output: {sample_result.image.shape}, {sample_result.image.dtype}")
    print(
        f"Calibrated spacing: dx={np.diff(sample_result.x_um).mean():.6f} um/px, "
        f"dy={np.diff(sample_result.y_um).mean():.6f} um/px"
    )

    original = _load_image(WHITE_LIGHT_PATH)
    brightened_original = _add_brightness_safely(original, 50.0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
    axes[0].imshow(brightened_original)
    axes[0].plot([224, 970], [89.256, 92.309], "c.-", linewidth=1)
    axes[0].set_title("Brightened original + orientation points")
    axes[1].imshow(sample_result.image)
    axes[1].set_title(f"Rotated by {sample_result.angle_deg:.3f} degrees")
    for axis in axes:
        axis.set_axis_off()
    plt.show()
else:
    print(f"Sample image is not available on this machine: {WHITE_LIGHT_PATH}")


## Verification

These assertions check sign convention, fixed output size, dtype preservation, saturation rather than uint8 overflow, calibrated pixel spacing, centering, and input immutability. The synthetic sloped line must become horizontal after the computed positive rotation.

In [ ]:
synthetic = np.zeros((121, 161), dtype=np.uint8)
line_x = np.arange(20, 141)
line_y = np.rint(40 + 0.2 * (line_x - 20)).astype(int)
synthetic[line_y, line_x] = 255
synthetic_before = synthetic.copy()

test_result = rotate_trench_image(
    synthetic,
    rotation_points_px=((line_x[0], line_y[0]), (line_x[-1], line_y[-1])),
    center_points_px=((40.0, 60.0), (80.0, 60.0)),
    brightness_offset=0,
    interpolation_order=0,
)
rotated_y, rotated_x = np.nonzero(test_result.image)
remaining_slope = np.polyfit(rotated_x, rotated_y, 1)[0]

assert test_result.angle_deg > 0
assert abs(remaining_slope) < 0.01, remaining_slope
assert test_result.image.shape == synthetic.shape
assert test_result.image.dtype == synthetic.dtype
assert np.array_equal(synthetic, synthetic_before), "The input array was mutated"
np.testing.assert_allclose(np.diff(test_result.x_um), 5.3 / 50.0, atol=1e-12)
np.testing.assert_allclose(np.diff(test_result.y_um), 5.3 / 50.0, atol=1e-12)
assert test_result.x_um[60] == 0.0
assert test_result.y_um[60] == 0.0

saturation_test = rotate_trench_image(
    np.full((5, 5), 250, dtype=np.uint8),
    angle_deg=0,
    brightness_offset=50,
)
assert np.all(saturation_test.image == 255), "uint8 brightness must saturate, not wrap"

if WHITE_LIGHT_PATH.is_file():
    assert sample_result.image.shape == (1024, 1280, 3)
    assert sample_result.image.dtype == np.uint8

print(f"All verification checks passed; residual synthetic slope = {remaining_slope:.6g}.")


## Embedding in a class

The core function does not rely on notebook state. A class can call it directly from a method, or expose it as a static method:

```python
class ImageProfiler:
    rotate_trench = staticmethod(rotate_trench_image)

result = ImageProfiler.rotate_trench(
    path,
    rotation_points_px=((x_left, y_left), (x_right, y_right)),
)
rotated_image = result.image
```

Use `result.save(path)` when an output file is desired. Keeping saving separate from rotation makes the computational method easier to test and reuse.